In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pytesseract
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
from time import sleep
import datetime
from bs4 import BeautifulSoup
from pandas import ExcelWriter
from PIL import Image
import re
import os
import xml.etree.ElementTree as ET
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\Tesseract-OCR\tesseract.exe"

#pytesseract.pytesseract.tesseract_cmd = r"C:\ProgramData\Tesseract-OCR\tesseract.exe"
    
    

In [2]:

#------------------------------------------------ Begin_ fileName ----------------------------------------
print("Running CZ CNBCZ Web Scraping Tool v.1.0")
print("***The program requires Tesseract, which path must be stated in the line 30 of this script. No longer update is required\n unless the regulator's site is considerably modified.***")
regulatorName = 'CZ CNBCZ'
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder, 'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

now=datetime.datetime.now()
filename= 'CZ CNBCZ Data Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
#writer = ExcelWriter(filename)
#------------------------------------------------ Begin_Varible ----------------------------------------

mainUrl  = 'https://apl.cnb.cz/apljerrsdad/JERRS.WEB15.BASIC_LISTINGS?p_lang=en'
regdict= {
	'CZ CNBCZ 1': '1', 
	'CZ CNBCZ 2': '16', 
	'CZ CNBCZ 3': '133', 
	'CZ CNBCZ 4': '134', 
	'CZ CNBCZ 5': '37', 
	'CZ CNBCZ 6': '140', 
	'CZ CNBCZ 7': '17', 
	'CZ CNBCZ 8': '31', 
	'CZ CNBCZ 9': '139', 
	'CZ CNBCZ 10': '53',
	'CZ CNBCZ 11': '239', 
	'CZ CNBCZ 12': '248', 
	'CZ CNBCZ 13': '82', 
	'CZ CNBCZ 14': '141', 
	'CZ CNBCZ 15': '235', 
	'CZ CNBCZ 16': '253', # Deal with alone
	'CZ CNBCZ 17': '223', 
	'CZ CNBCZ 18': '226', 
	'CZ CNBCZ 20': '224', 
	'CZ CNBCZ 21': '231', 
	'CZ CNBCZ 22': '174', 
	'CZ CNBCZ 23': '184', 
	'CZ CNBCZ 24': '182', 
	'CZ CNBCZ 25': '183', 
	'CZ CNBCZ 26': '205', 
	'CZ CNBCZ 27': '206', 
	'CZ CNBCZ 28': '118'
	}

Typology = {
	'CZ CNBCZ 1': 'Banks and branches of foreign banks', 
	'CZ CNBCZ 2': 'Credit unions', 
	'CZ CNBCZ 3': 'Representative office of foreign bank', 
	'CZ CNBCZ 4': 'Foreign financial and credit institutions and branches of foreign financial and credit institutions providing cross-border services in the Czech Republic', 
	'CZ CNBCZ 5': 'Investment firms incl. branches of foreign investment firms', 
	'CZ CNBCZ 6': 'Foreign investment firms providing cross-border services in the Czech Republic', 
	'CZ CNBCZ 7': 'Insurance companies and branches of foreign Insurance companies', 
	'CZ CNBCZ 8': 'Reinsurers', 
	'CZ CNBCZ 9': 'Foreign insurance companies and branches of foreign insurance companies providing cross-border services in the Czech Republic', 
	'CZ CNBCZ 10': 'Management companies and branches of foreign management companies',
	'CZ CNBCZ 11': 'Investment funds depositaries', 
	'CZ CNBCZ 12': 'Investment funds with legal personality', 
	'CZ CNBCZ 13': 'Common funds (constituted in accordance with contract law)', 
	'CZ CNBCZ 14': 'Foreign management companies from EU providing cross-border services in the Czech Republic', 
	'CZ CNBCZ 15': 'Fund managers with registered office in foreign country authorized to manage investment funds', 
	'CZ CNBCZ 16': 'Foreign investment funds, investments in which can be offerred in the Czech Republic', 
	'CZ CNBCZ 17': 'Pension management companies', 
	'CZ CNBCZ 18': 'Participation funds', 
	'CZ CNBCZ 20': 'Transformed (pension) funds', 
	'CZ CNBCZ 21': 'Foreign institutions for occupational retirement provision providing cross-border services in the Czech Republic', 
	'CZ CNBCZ 22': 'Payment institutions and foreign payment institutions established in the Czech Republic', 
	'CZ CNBCZ 23': 'Small payment institutions', 
	'CZ CNBCZ 24': 'Electronic money institutions and foreign electronic money institutions established in the Czech Republic', 
	'CZ CNBCZ 25': 'Small e-money issuers', 
	'CZ CNBCZ 26': 'Foreign payment institutions providing cross-border services in the Czech Republic', 
	'CZ CNBCZ 27': 'Foreign electronic money institutions providing cross-border services in the Czech Republic', 
	'CZ CNBCZ 28': 'Bureaux de change'	

}


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
'Address_1 - Mother company': [], 'Address_2 - Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
'Phone - Mother company': [], 'Check': []}

catlist = ['Name', 'RECNTRY', 'REGCODE', 'LISTCODE', 'Entity Type', 'Personal identification number', 'Institution Name', 'Registered / permanent residence address', 'Contact Address', 'Phone', 'Fax', 'E-mail', 'Website', 'Numeric code', 'LEI', 'Type of authorization', 'Date of authorization', 'Date the decision came to legal force', 'Ownership Structure', 'Related legal ties', 'Other function(s)', 'Note', 'Head office', 'Detailed Entity Type', 'Branches and subsidiaries abroad', 'Cross-border services', 'Date of entry in the Companies register', 'Date of entry in the list', 'Depository', 'Management company', 'Cross-border marketing', 'EU management company', 'Local contact person', 'Registration number', 'Concession / Registration number']

label_sql = {'Entity Type' : 'Typology', 'Personal identification number' : 'InternalID_1', 'Registered / permanent residence address' : 'Address_1', 'Phone' : 'Phone', 'Fax' :'Fax', 'E-mail' : 'Email', 'Website' : 'Website', 'Numeric code' : 'InternalID_2', 'LEI' : 'LEI Code', 'Type of authorization' : 'RegulationType', 'Date of authorization' : 'RegulationDate', 'Head office' : 'Name - Mother Company'}

processdate = now.strftime('%Y-%m-%d')


Running CZ CNBCZ Web Scraping Tool v.1.0
***The program requires Tesseract, which path must be stated in the line 30 of this script. No longer update is required
 unless the regulator's site is considerably modified.***


In [3]:

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict
def get_captcha(file_path):
    captcha = pytesseract.image_to_string(file_path)
    print('1. OCRed text: ',captcha)
    captcha_answer = ''.join([char for char in captcha if char.isdigit()])
    print('2. Captcha: ',captcha_answer)
    return captcha_answer






In [4]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

In [5]:
#------------------------------------------------ Main Function ----------------------------------------

driver.get(mainUrl)
sleep(3)
image_path = os.path.join(tempfolder, 'captcha.png')
image = driver.find_element(By.ID, 'ID_EMAIL_FORM').screenshot(image_path)
#driver.find_element(By.XPATH, '//*[@id="OPIS"]').click() #clicking on the captcha field input field
driver.find_element(By.XPATH, '//*[@id="OPIS"]').send_keys(get_captcha(image_path)) #Sending captcha answer
os.remove(image_path)
sleep(1)
sendcap = driver.find_element(By.XPATH, '//input[@value="Confirm"]').click()#Confirming captcha
sleep(2)
step0 = driver.find_element(By.LINK_TEXT, 'Predefined lists').click()
sleep(2)
step1 = driver.find_element(By.XPATH, '//*[@id="ID_REC_PER_PAGE"]/option[@value="108"]').click() # list value 108
sleep(2)
step2 = driver.find_element(By.XPATH, '//input[@value="Next step(s)"]').click()
sleep(2)
soup = BeautifulSoup(driver.page_source, 'html.parser')
# list_urls = soup.find('table').find_all('a')
# print(len(list_urls))

table_info = ''
for reg in regdict:
    print(Typology[reg])
    sleep(2)
    # link = driver.find_element(By.LINK_TEXT, Typology[reg])
    # sleep(2)
    # driver.execute_script("arguments[0].click();", link) 
    url_ ='https://jerrs.cnb.cz/apljerrsdad/JERRS.WEB15.BASIC_LISTINGS_RESPONSE_3?p_lang=en&p_DATUM=07.11.2025&p_hie=HI&p_rec_per_page=108&p_ses_idx='+regdict[reg]
    
    sleep(2)
    driver.switch_to.new_window('tab')         # open a fresh tab
    driver.get(url_)                           # load data there
    sleep(2)
    # Switch to the new tab
    window_handles = driver.window_handles
    # driver.switch_to.window(window_handles[-1])
    # if reg != 'CZ CNBCZ 16':

    #     # Locate an element in the new tab
    #     element = driver.find_element(By.ID, "nadpisStranky")
    #     tbody = driver.find_element(By.TAG_NAME,'table').find_element(By.TAG_NAME,'tbody')
    #     rows = tbody.find_elements(By.TAG_NAME, "tr")

    #     for row in rows:
    #         cells = row.find_elements(By.CSS_SELECTOR, "td.tableDetail")
    #         texts = [cell.text.strip() for cell in cells]

    #         if not texts:  # skip header/empty rows
    #             continue

    #         internal_id = texts[0]
    #         #print(internal_id)
    #         name = texts[1]
    #         address = texts[2]
    #         city = texts[3]
    #         zip_ = texts[4]
    #         cntry = texts[-2]
    #         date_from = texts[-1]


    #         sqldict['Name'].append(name)
    #         sqldict['Address_1'].append(address)
    #         sqldict['InternalID_1'].append(internal_id)
    #         sqldict['InternalID_1_type'].append('Personal identification number')
    #         sqldict['ListProcessDate'].append(processdate)
    #         sqldict['City'].append(city)
    #         sqldict['Zip'].append(zip_)
    #         sqldict['Cntry'].append(cntry)
    #         sqldict['RegulationDate'].append(date_from)
    #         sqldict['ListName'].append(Typology[reg])
    #         sqldict['RegulationType'].append('Regulated')
    #         sqldict['RegCtry'].append(reg.split(' ')[0])
    #         sqldict['RegCode'].append(reg.split(' ')[1])
    #         sqldict['ListCode'].append(reg.split(' ')[-1])
    #         sqldict = bourange_same_length_array(sqldict)

    # else:
    sleep(5)
    
    dowm_load_xml_btn = driver.find_element(By.LINK_TEXT,'Download entities in XML format')
    try:
        sleep(4)
        dowm_load_xml_btn.click()
    except Exception as e:
        raise(f"Error clicking download button: {Typology[reg],e}")
    sleep(5)
    excel_file = os.listdir(tempfolder)[0]
    filePath = os.path.join(tempfolder, excel_file)
    # Load XML file
    tree = ET.parse(filePath)
    root = tree.getroot()

    # Extract records
    records = []
    for rec in root.findall('D-Rec'):
        record = {
            'E-Icn': rec.findtext('E-Icn'),
            'F-IcnT': rec.findtext('F-IcnT'),
            'G-Ident': rec.findtext('G-Ident'),
            'H-Addr': rec.findtext('H-Addr'),
            'I-City': rec.findtext('I-City'),
            'J-PostC': rec.findtext('J-PostC'),
            'K-CountC': rec.findtext('K-CountC'),
            'L-DateFrom': rec.findtext('L-DateFrom')
        }
        records.append(record)

    # Convert to DataFrame
    df = pd.DataFrame(records)
    sleep(1)
    if os.path.exists(tempfolder):

        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))

    else:

        os.mkdir(tempfolder)


    # find first row where both 'E-Icn' and 'F-IcnT' are non-empty (not NaN and not empty string)
    valid_rows = df[
        df['E-Icn'].notna() & (df['E-Icn'].astype(str).str.strip() != '') &
        df['F-IcnT'].notna() & (df['F-IcnT'].astype(str).str.strip() != '')
    ]
    if valid_rows.empty:
        raise ValueError("No rows with non-empty 'E-Icn' and 'F-IcnT' found.")
    start_idx = valid_rows.index[0]
    # Loop through rows and cells
    for i in range(start_idx+1, len(df)):
        for col in df.columns:
            value = df.iloc[i][col]
            #print(f"Row {i}, Column '{col}': {value}")
            if col == 'G-Ident':
                    name = value
                    # print(name)
                    sqldict['Name'].append(name)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                
            elif col == 'E-Icn':
                internal_id = value
                sqldict['InternalID_1'].append(internal_id)
                
            elif col == 'F-IcnT':
                internal_id_type = value
                sqldict['InternalID_1_type'].append(internal_id_type)

            elif col == 'H-Addr':
                address = value
                sqldict['Address_1'].append(address)
            elif col == 'I-City':
                city = value
                sqldict['City'].append(city)
            elif col == 'J-PostC':
                zip_ = value
                sqldict['Zip'].append(zip_)
            elif col == 'K-CountC':
                cntry = value
                sqldict['Cntry'].append(cntry)
            elif col == 'L-DateFrom':
                date_from = value
                sqldict['RegulationDate'].append(date_from)
        sqldict = bourange_same_length_array(sqldict)
                    
    # Close the new tab
    driver.close()
    # Switch back to the original tab (List page)
    driver.switch_to.window(window_handles[0])



1. OCRed text:  326729

(Confirm)

‘Type the

code:


2. Captcha:  326729
Banks and branches of foreign banks
Credit unions
Representative office of foreign bank
Foreign financial and credit institutions and branches of foreign financial and credit institutions providing cross-border services in the Czech Republic
Investment firms incl. branches of foreign investment firms
Foreign investment firms providing cross-border services in the Czech Republic
Insurance companies and branches of foreign Insurance companies
Reinsurers
Foreign insurance companies and branches of foreign insurance companies providing cross-border services in the Czech Republic
Management companies and branches of foreign management companies
Investment funds depositaries
Investment funds with legal personality
Common funds (constituted in accordance with contract law)
Foreign management companies from EU providing cross-border services in the Czech Republic
Fund managers with registered office in foreign country au

In [6]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
df=pd.DataFrame(sqldict)

df = df[df['Name']!='']
df.to_excel(filename, index=False)
sleep(3)

driver.quit()

In [7]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
1,,,,,,"AS Inbank, odštěpný závod",14028999,ICO,,,...,,,,,,,,,,
2,,,,,,Air Bank a.s.,29045371,ICO,,,...,,,,,,,,,,
3,,,,,,"BNP Paribas S.A., pobočka Česká republika",06325416,ICO,,,...,,,,,,,,,,
4,,,,,,"Bank Gutmann Aktiengesellschaft, pobočka Česká...",24131768,ICO,,,...,,,,,,,,,,
5,,,,,,Bank of China (CEE) Ltd. Prague Branch,04253434,ICO,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7429,,,,,,"Šustr Marek, Ing.",69874999,ICO,,,...,,,,,,,,,,
7430,,,,,,Švach Jan,18092462,ICO,,,...,,,,,,,,,,
7431,,,,,,Šverma Josef,86715895,ICO,,,...,,,,,,,,,,
7432,,,,,,Šípal Ladislav,44574592,ICO,,,...,,,,,,,,,,
